In this lab, you'll explore the powerful capabilities of tool calling in large language models (LLMs) to build advanced AI agents that can dynamically interact with users. Using the LangChain framework, you’ll learn how to build an interactive agent that responds to user queries by selecting and executing the right function at the right time. This hands-on approach will help you understand how LLMs can be extended with real-world functionality, bridging natural language understanding with dynamic, tool-based actions.


<h2>Objectives</h2>
After completing this lab you will be able to:

Initialize a chat model for tool interactions
Define and bind custom tools to the LLM for expanded functionality
Use mapping dictionaries for dynamic function calls
Extract tool names and functions for precise function calls
Build agent classes that manage the entire tool-calling process

<h2>Setup</h2>
For this lab, you will be using the following libraries:

langchain is the framework you will build the agent on.
langchain-openai is a partner package of LangChain and integrates OpenAI LLMs to the framework.

In [ ]:
%pip install langchain===0.3.25
%pip install langchain-openai===0.3.19

In [ ]:
%pip install langchain-ollama==0.3.10

In [3]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

In [13]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("llama3.1:8b", model_provider="ollama")

In [6]:
@tool
def add(a: int, b: int) -> int:
    """
    Add a and b.
    
    Args:
        a (int): first integer to be added
        b (int): second integer to be added

    Return:
        int: sum of a and b
    """
    return a + b

In [7]:
tools = [add]

llm_with_tools = llm.bind_tools(tools)

In [8]:
@tool
def subtract(a: int, b:int) -> int:
    """Subtract b from a."""
    return a - b

@tool
def multiply(a: int, b:int) -> int:
    """Multiply a and b."""
    return a * b


In [9]:
tool_map = {
    "add": add, 
    "subtract": subtract,
    "multiply": multiply
}

input_ = {
    "a": 1,
    "b": 2
}

tool_map["add"].invoke(input_)

3

In [14]:
tools = [add, subtract, multiply]

llm_with_tools = llm.bind_tools(tools)

query = "What is 3 + 2?"
chat_history = [HumanMessage(content=query)]

response_1 = llm_with_tools.invoke(chat_history)
chat_history.append(response_1)

print(type(response_1))
#print(response_1)

<class 'langchain_core.messages.ai.AIMessage'>


In [15]:
tool_calls_1 = response_1.tool_calls

tool_1_name = tool_calls_1[0]["name"]
tool_1_args = tool_calls_1[0]["args"]
tool_call_1_id = tool_calls_1[0]["id"]

print(f'tool name:\n{tool_1_name}')
print(f'tool args:\n{tool_1_args}')
print(f'tool call ID:\n{tool_call_1_id}')

tool name:
add
tool args:
{'a': 3, 'b': 2}
tool call ID:
7c37a80d-9c49-4a66-9853-48d60200ab81


In [16]:
tool_response = tool_map[tool_1_name].invoke(tool_1_args)
tool_message = ToolMessage(content=tool_response, tool_call_id=tool_call_1_id)

print(tool_message)

content='5' tool_call_id='7c37a80d-9c49-4a66-9853-48d60200ab81'


In [17]:
chat_history.append(tool_message) # ignore any syntax error highlight in VS code

answer = llm_with_tools.invoke(chat_history)
print(type(answer))
print(answer.content)

<class 'langchain_core.messages.ai.AIMessage'>
The final answer is 5.


In [18]:
class ToolCallingAgent:
    def __init__(self, llm):
        self.llm_with_tools = llm.bind_tools(tools)
        self.tool_map = tool_map

    def run(self, query: str) -> str:
        # Step 1: Initial user message
        chat_history = [HumanMessage(content=query)]

        # Step 2: LLM chooses tool
        response = self.llm_with_tools.invoke(chat_history)
        if not response.tool_calls:
            return response.contet # Direct response, no tool needed
        # Step 3: Handle first tool call
        tool_call = response.tool_calls[0]
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_call_id = tool_call["id"]

        # Step 4: Call tool manually
        tool_result = self.tool_map[tool_name].invoke(tool_args)

        # Step 5: Send result back to LLM
        tool_message = ToolMessage(content=str(tool_result), tool_call_id=tool_call_id)
        chat_history.extend([response, tool_message])

        # Step 6: Final LLM result
        final_response = self.llm_with_tools.invoke(chat_history)
        return final_response.content

In [19]:
my_agent = ToolCallingAgent(llm)

print(my_agent.run("one plus 2"))

print(my_agent.run("one - 2"))

print(my_agent.run("three times two"))

The answer to the original user question is 3.
The result of subtracting 2 from 1 is -1.
The output of the tool call is 6, so the answer to the original user question is:

"Three times two is 6."
